# Glue job: bronze to silver

Deploys the Glue job `pf-bronze-to-silver`, runs it, and checks the five silver tables against the source files.
The job reads one batch from `bronze/patient_records/ingest_date=<date>/` and writes Parquet tables to `silver/`,
registered in the Glue database `pf_silver`.

Run the cells top to bottom. Each cell is safe to run twice, and prints the AWS CLI command it runs (`$ aws ...`).
Each job run takes a few minutes and costs a few cents.

## 1. Setup

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI and prints the command and its output (see infra/aws_helpers.py)
from aws_helpers import SETTINGS, LAKE, aws, aws_json, run_athena, run_glue_job, whoami

JOB = SETTINGS["glue_job_name"]                      # pf-bronze-to-silver
SILVER_DB = SETTINGS["glue_databases"]["silver"]     # pf_silver
ROLE = "pf-glue-role"
BUILD = ROOT / "build"
BUILD.mkdir(exist_ok=True)

print("identity:", whoami())
print("job     :", JOB)
print("database:", SILVER_DB)

identity: user/Ahmed
job     : pf-bronze-to-silver
database: pf_silver


## 2. Upload the scripts
Glue reads the job script from S3. `fhir_utils.py` is uploaded next to it and passed to the job with `--extra-py-files`,
so `import fhir_utils` works inside the job.

In [2]:
for script in ("bronze_to_silver.py", "fhir_utils.py"):
    aws("s3", "cp", f"glue_jobs/{script}", f"s3://{LAKE}/glue_scripts/{script}", "--only-show-errors")

uploaded = aws_json("s3api", "list-objects-v2", "--bucket", LAKE, "--prefix", "glue_scripts/")["Contents"]
print({obj["Key"]: obj["Size"] for obj in uploaded})
assert {obj["Key"] for obj in uploaded} == {"glue_scripts/bronze_to_silver.py", "glue_scripts/fhir_utils.py"}

$ aws s3 cp glue_jobs/bronze_to_silver.py s3://patientflow-datalake-zhmkbf/glue_scripts/bronze_to_silver.py --only-show-errors


$ aws s3 cp glue_jobs/fhir_utils.py s3://patientflow-datalake-zhmkbf/glue_scripts/fhir_utils.py --only-show-errors


{'glue_scripts/bronze_to_silver.py': 5593, 'glue_scripts/fhir_utils.py': 11217}


## 3. Create or update the job
| Setting | Value | Why |
|---|---|---|
| Glue version | 5.0 | Spark 3.5.4, Python 3.11 (the same Spark version the local tests use) |
| Workers | 2 x G.1X | the smallest sensible size: 2 DPU (1 DPU = 4 vCPU, 16 GB memory) |
| Timeout | 10 minutes | a stuck run cannot keep billing |
| Retries | 0 | a failure stops at once instead of paying for a second run |
| `--enable-glue-datacatalog` | true | lets `spark.sql` create tables in the Glue Data Catalog |
| `--extra-py-files` | `fhir_utils.py` on S3 | the tested flattening functions |

In [3]:
role_arn = aws_json("iam", "get-role", "--role-name", ROLE)["Role"]["Arn"]

JOB_SETTINGS = {
    "Role": role_arn,
    "Command": {"Name": "glueetl", "ScriptLocation": f"s3://{LAKE}/glue_scripts/bronze_to_silver.py", "PythonVersion": "3"},
    "GlueVersion": "5.0",
    "WorkerType": "G.1X",
    "NumberOfWorkers": 2,
    "Timeout": 10,
    "MaxRetries": 0,
    "DefaultArguments": {
        "--extra-py-files": f"s3://{LAKE}/glue_scripts/fhir_utils.py",
        "--enable-glue-datacatalog": "true",
        "--lake_bucket": LAKE,
        "--silver_database": SILVER_DB,
        "--TempDir": f"s3://{LAKE}/temp/",
        "--job-language": "python",
    },
}

exists = aws("glue", "get-job", "--job-name", JOB, check=False, show=False).returncode == 0
if exists:
    print(f"job {JOB} already exists - updating it\n")
    aws("glue", "update-job", "--job-name", JOB, "--job-update", json.dumps(JOB_SETTINGS),
        "--query", "JobName", "--output", "text")
else:
    aws("glue", "create-job", "--cli-input-json",
        json.dumps({"Name": JOB, "Tags": {"project": "patient-flow"}, **JOB_SETTINGS}),
        "--query", "Name", "--output", "text")

$ aws glue get-job --job-name pf-bronze-to-silver


job pf-bronze-to-silver already exists - updating it

$ aws glue update-job --job-name pf-bronze-to-silver --job-update {"Role": "arn:aws:iam::<ACCOUNT_ID>:role/pf-glue-role", "Command": {"Name": "glueetl", "ScriptLocation": "s3://patientflow-datalake-zhmkbf/glue_scripts/bronze_to_silver.py", "PythonVersion": "3"}, "GlueVersion": "5.0", "WorkerType": "G.1X", "NumberOfWorkers": 2, "Timeout": 10, "MaxRetries": 0, "DefaultArguments": {"--extra-py-files": "s3://patientflow-datalake-zhmkbf/glue_scripts/fhir_utils.py", "--enable-glue-datacatalog": "true", "--lake_bucket": "patientflow-datalake-zhmkbf", "--silver_database": "pf_silver", "--TempDir": "s3://patientflow-datalake-zhmkbf/temp/", "--job-language": "python"}} --query JobName --output text


pf-bronze-to-silver


In [4]:
job = aws_json("glue", "get-job", "--job-name", JOB)["Job"]
assert job["GlueVersion"] == "5.0" and job["WorkerType"] == "G.1X" and job["NumberOfWorkers"] == 2
assert job["Timeout"] == 10 and job["MaxRetries"] == 0 and job["Role"] == role_arn
assert job["DefaultArguments"]["--enable-glue-datacatalog"] == "true"
print(f"OK  {JOB}: Glue {job['GlueVersion']}, {job['NumberOfWorkers']} x {job['WorkerType']}, "
      f"{job['Timeout']} min timeout, {job['MaxRetries']} retries")

OK  pf-bronze-to-silver: Glue 5.0, 2 x G.1X, 10 min timeout, 0 retries


## 4. What the source files contain
The expected row counts are computed from the local batch files with plain Python, independently of the Spark code.
They are what the silver tables must contain.

In [5]:
def read_source(batch):
    """Counts and ids per resource type found in the local files of one batch."""
    found = {"Patient": [], "Encounter": [], "Condition": [], "Organization": [], "Practitioner": []}
    for path in sorted((ROOT / "data" / "landing" / f"batch_{batch:02d}").glob("*.json")):
        for entry in json.loads(path.read_text(encoding="utf-8"))["entry"]:
            resource = entry["resource"]
            if resource["resourceType"] in found:
                found[resource["resourceType"]].append(resource)
    return found


EXPECTED_TABLE = {"patient": "Patient", "encounter": "Encounter", "condition": "Condition",
                  "organization": "Organization", "practitioner": "Practitioner"}
source_1 = read_source(1)
print({name: len(items) for name, items in source_1.items()})

{'Patient': 35, 'Encounter': 1411, 'Condition': 854, 'Organization': 521, 'Practitioner': 521}


## 5. Run the job for batch 1 (`ingest_date` 2026-11-01)
The bronze folder for this date was filled by the ingest Lambda. The job lists it, builds the tables, checks them,
then writes each table and registers its partition.

In [6]:
run_1 = run_glue_job(JOB, {"--ingest_date": "2026-11-01"})
print(run_1)
assert run_1["state"] == "SUCCEEDED", run_1["error"]

# Glue bills DPU-seconds; 1 DPU costs about $0.44 per hour in us-east-1 (check current pricing)
print(f"ran {run_1['execution_seconds']} s, {run_1['dpu_seconds']} DPU-seconds, about ${run_1['dpu_seconds'] / 3600 * 0.44:.3f}")

started pf-bronze-to-silver run jr_1a1701bebfe20104da8551ba6e859b9838a6250ba7b9ffd8b23c68a45e7dc087


  19:38:49  RUNNING


  19:40:59  SUCCEEDED
{'run_id': 'jr_1a1701bebfe20104da8551ba6e859b9838a6250ba7b9ffd8b23c68a45e7dc087', 'state': 'SUCCEEDED', 'error': None, 'execution_seconds': 121, 'dpu_seconds': 242.0}
ran 121 s, 242.0 DPU-seconds, about $0.030


## 6. Check the silver tables
### Row counts against the source

In [7]:
def silver_count(table, ingest_date):
    result = run_athena(f"SELECT COUNT(*) FROM {SILVER_DB}.{table} WHERE ingest_date = '{ingest_date}'")
    return int(result["rows"][0][0])


for table, resource_type in EXPECTED_TABLE.items():
    expected, actual = len(source_1[resource_type]), silver_count(table, "2026-11-01")
    assert actual == expected, f"{table}: silver has {actual} rows, source has {expected}"
    print(f"OK  {table:<13} {actual:>6} rows = source")

OK  patient           35 rows = source


OK  encounter       1411 rows = source


OK  condition        854 rows = source


OK  organization     521 rows = source


OK  practitioner     521 rows = source


### The same rows, not only the same counts
The ids in silver must be exactly the ids in the source files.

In [8]:
ID_COLUMN = {"patient": "patient_id", "encounter": "encounter_id", "condition": "condition_id",
             "organization": "organization_id", "practitioner": "practitioner_id"}

for table, resource_type in EXPECTED_TABLE.items():
    ids = {row[0] for row in run_athena(
        f"SELECT {ID_COLUMN[table]} FROM {SILVER_DB}.{table} WHERE ingest_date = '2026-11-01'")["rows"]}
    assert ids == {r["id"] for r in source_1[resource_type]}, f"{table}: ids differ from the source"
    print(f"OK  {table:<13} {len(ids):>6} ids identical to the source")

OK  patient           35 ids identical to the source


OK  encounter       1411 ids identical to the source


OK  condition        854 ids identical to the source


OK  organization     521 ids identical to the source


OK  practitioner     521 ids identical to the source


### Column types
Athena shows Glue's `string` as `varchar` and a Glue `timestamp` as `timestamp(3)`.

In [9]:
columns = run_athena(
    "SELECT table_name, column_name, data_type FROM information_schema.columns "
    f"WHERE table_schema = '{SILVER_DB}' ORDER BY table_name, ordinal_position")["rows"]

schema = {}
for table, column, dtype in columns:
    schema.setdefault(table, {})[column] = dtype

for table in ("patient", "encounter", "condition", "organization", "practitioner"):
    print(f"{table}: " + ", ".join(f"{c} {t}" for c, t in schema[table].items()))

assert schema["patient"]["birth_date"] == "date"
assert schema["patient"]["deceased_datetime"].startswith("timestamp")
assert schema["encounter"]["start_ts"].startswith("timestamp") and schema["encounter"]["end_ts"].startswith("timestamp")
assert schema["condition"]["onset_ts"].startswith("timestamp")
assert all(schema[t]["ingest_date"] == "varchar" for t in schema)
assert "practitioner_npi" in schema["encounter"]
print("\nOK  types are as designed (dates, timestamps, ingest_date partition)")

patient: patient_id varchar, gender varchar, birth_date date, deceased_datetime timestamp(3), marital_status varchar, city varchar, state varchar, postal_code varchar, ingest_date varchar
encounter: encounter_id varchar, patient_id varchar, status varchar, encounter_class varchar, type_code varchar, type_display varchar, start_ts timestamp(3), end_ts timestamp(3), organization_id varchar, practitioner_npi varchar, reason_code varchar, reason_display varchar, ingest_date varchar
condition: condition_id varchar, patient_id varchar, encounter_id varchar, code varchar, display varchar, clinical_status varchar, onset_ts timestamp(3), abatement_ts timestamp(3), ingest_date varchar
organization: organization_id varchar, name varchar, type varchar, city varchar, state varchar, postal_code varchar, ingest_date varchar
practitioner: practitioner_id varchar, npi varchar, full_name varchar, gender varchar, ingest_date varchar

OK  types are as designed (dates, timestamps, ingest_date partition)


### One real encounter, field by field
Take an inpatient encounter from the source file, work out what each silver column should be with plain Python
(IDs from the references, timestamps converted to UTC), and compare with what Athena returns.

In [10]:
def to_utc_text(iso_text):
    return datetime.fromisoformat(iso_text).astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


sample = next(e for e in source_1["Encounter"] if e["class"]["code"] == "IMP")
expected = {
    "encounter_id": sample["id"],
    "patient_id": sample["subject"]["reference"].removeprefix("urn:uuid:"),
    "encounter_class": "IMP",
    "start_ts": to_utc_text(sample["period"]["start"]),
    "end_ts": to_utc_text(sample["period"]["end"]),
    "organization_id": sample["serviceProvider"]["reference"].rsplit("|", 1)[1],
    "practitioner_npi": sample["participant"][0]["individual"]["reference"].rsplit("|", 1)[1],
    "reason_code": sample["reasonCode"][0]["coding"][0]["code"],
}
result = run_athena(
    "SELECT encounter_id, patient_id, encounter_class, "
    "date_format(start_ts, '%Y-%m-%d %H:%i:%s'), date_format(end_ts, '%Y-%m-%d %H:%i:%s'), "
    "organization_id, practitioner_npi, reason_code "
    f"FROM {SILVER_DB}.encounter WHERE ingest_date = '2026-11-01' AND encounter_id = '{sample['id']}'")
actual = dict(zip(expected, result["rows"][0], strict=True))

for column in expected:
    print(f"{column:<17} source: {expected[column]:<40} silver: {actual[column]}")
assert actual == expected
print("\nOK  silver matches the source for every checked column (start 08:10 +05:00 is stored as 03:10 UTC)")

encounter_id      source: 2c588417-7537-a68c-0ec1-f4fa3d2d5f6e     silver: 2c588417-7537-a68c-0ec1-f4fa3d2d5f6e
patient_id        source: 2c588417-7537-a68c-7c2d-e1a45847fcc9     silver: 2c588417-7537-a68c-7c2d-e1a45847fcc9
encounter_class   source: IMP                                      silver: IMP
start_ts          source: 2004-10-16 01:22:33                      silver: 2004-10-16 01:22:33
end_ts            source: 2004-10-19 06:17:58                      silver: 2004-10-19 06:17:58
organization_id   source: 9adfae03-673d-365d-9052-f705098fa3e4     silver: 9adfae03-673d-365d-9052-f705098fa3e4
practitioner_npi  source: 9999990093                               silver: 9999990093
reason_code       source: 698306007                                silver: 698306007

OK  silver matches the source for every checked column (start 08:10 +05:00 is stored as 03:10 UTC)


### Keys that must join
Every encounter must point to a real organization, practitioner (by NPI), patient; every condition to a real encounter.

In [11]:
checks = {
    "encounters without a patient": f"SELECT COUNT(*) FROM {SILVER_DB}.encounter e LEFT JOIN {SILVER_DB}.patient p ON e.patient_id = p.patient_id AND p.ingest_date = e.ingest_date WHERE e.ingest_date = '2026-11-01' AND p.patient_id IS NULL",
    "encounters without an organization": f"SELECT COUNT(*) FROM {SILVER_DB}.encounter e LEFT JOIN {SILVER_DB}.organization o ON e.organization_id = o.organization_id WHERE e.ingest_date = '2026-11-01' AND o.organization_id IS NULL",
    "encounters without a practitioner (by NPI)": f"SELECT COUNT(*) FROM {SILVER_DB}.encounter e LEFT JOIN {SILVER_DB}.practitioner p ON e.practitioner_npi = p.npi WHERE e.ingest_date = '2026-11-01' AND p.npi IS NULL",
    "conditions without an encounter": f"SELECT COUNT(*) FROM {SILVER_DB}.condition c LEFT JOIN {SILVER_DB}.encounter e ON c.encounter_id = e.encounter_id AND e.ingest_date = c.ingest_date WHERE c.ingest_date = '2026-11-01' AND e.encounter_id IS NULL",
}
for label, sql in checks.items():
    orphans = int(run_athena(sql)["rows"][0][0])
    assert orphans == 0, f"{label}: {orphans}"
    print(f"OK  0 {label}")

OK  0 encounters without a patient


OK  0 encounters without an organization


OK  0 encounters without a practitioner (by NPI)


OK  0 conditions without an encounter


### Where the files are
Each job run writes Parquet files named `part-NNNNN-<write id>-c000.snappy.parquet`. All files written by one run
share the same write id, so the write id tells us which run produced a partition's files.
(The empty `<table>_$folder$` objects next to the table folders are placeholders created by the Glue file system;
Athena ignores them.)

In [12]:
def silver_objects():
    objects = aws_json("s3api", "list-objects-v2", "--bucket", LAKE, "--prefix", "silver/")["Contents"]
    return [obj for obj in objects if obj["Key"].endswith(".parquet")]


def write_ids(objects):
    """For each (table, partition), the set of write ids found in the file names."""
    ids = {}
    for obj in objects:
        table, partition, name = obj["Key"].split("/")[1:4]
        ids.setdefault((table, partition), set()).add("-".join(name.split("-")[2:7]))
    return ids


objects = silver_objects()
by_table = {}
for obj in objects:
    table = obj["Key"].split("/")[1]
    by_table[table] = by_table.get(table, 0) + obj["Size"]
for table, size in by_table.items():
    print(f"silver/{table:<13} {size / 1e6:6.2f} MB")

ids_after_first_run = write_ids(objects)
assert all(len(found) == 1 for found in ids_after_first_run.values()), "a partition mixes files of several runs"
print("example key:", objects[0]["Key"])
print(f"OK  {len(ids_after_first_run)} partitions, each written by exactly one run")

silver/condition       0.08 MB
silver/encounter       0.11 MB
silver/organization    0.05 MB
silver/patient         0.01 MB
silver/practitioner    0.04 MB
example key: silver/condition/ingest_date=2026-11-01/part-00000-7ec6804e-abed-4e33-93a1-054b96b4a212-c000.snappy.parquet
OK  5 partitions, each written by exactly one run


## 7. Run it again: same result
Running the same batch again must replace the partition. Same row counts, and the files must all come from the new run:
one write id per partition, different from the first run's (so nothing old is left and nothing is doubled).

In [13]:
run_2 = run_glue_job(JOB, {"--ingest_date": "2026-11-01"})
assert run_2["state"] == "SUCCEEDED", run_2["error"]

for table, resource_type in EXPECTED_TABLE.items():
    assert silver_count(table, "2026-11-01") == len(source_1[resource_type]), table

ids_after_rerun = write_ids(silver_objects())
assert ids_after_rerun.keys() == ids_after_first_run.keys()
for partition, found in ids_after_rerun.items():
    assert len(found) == 1, f"{partition}: files of more than one run"
    assert found != ids_after_first_run[partition], f"{partition}: not rewritten"
print("OK  second run: same row counts, every partition rewritten by the new run only")

started pf-bronze-to-silver run jr_9cdab10012becc12159ad57c9a29629f218543686e20811057baac8b6fe36d7d


  19:42:39  RUNNING


  19:44:51  SUCCEEDED


OK  second run: same row counts, every partition rewritten by the new run only


## 8. Batch 2: no hospital or practitioner files
Only batch 1 carries the hospital and practitioner lists. Batch 2 must still work: patients, encounters and
conditions are written, and the organization and practitioner tables are left alone.
First the Lambda ingests batch 2 into bronze.

In [14]:
out_file = BUILD / "invoke_result.json"
meta = json.loads(aws("lambda", "invoke", "--function-name", SETTINGS["lambda_name"],
                      "--payload", json.dumps({"batch": 2}), "--cli-binary-format", "raw-in-base64-out",
                      str(out_file.relative_to(ROOT))).stdout)
ingest_result = json.loads(out_file.read_text())
assert "FunctionError" not in meta and ingest_result["status"] == "OK", ingest_result
print(ingest_result)

$ aws lambda invoke --function-name pf-ingest --payload {"batch": 2} --cli-binary-format raw-in-base64-out build\invoke_result.json


{
    "StatusCode": 200,
    "ExecutedVersion": "$LATEST"
}
{'status': 'OK', 'batch': 2, 'ingest_date': '2026-11-02', 'file_count': 34}


In [15]:
run_3 = run_glue_job(JOB, {"--ingest_date": "2026-11-02"})
assert run_3["state"] == "SUCCEEDED", run_3["error"]

source_2 = read_source(2)
assert not source_2["Organization"] and not source_2["Practitioner"], "batch 2 was expected to have no reference files"
for table in ("patient", "encounter", "condition"):
    expected, actual = len(source_2[EXPECTED_TABLE[table]]), silver_count(table, "2026-11-02")
    assert actual == expected, f"{table}: {actual} vs {expected}"
    print(f"OK  {table:<13} {actual:>6} rows = source (batch 2)")

for table in ("organization", "practitioner"):
    partitions = [row[0] for row in run_athena(f'SELECT DISTINCT ingest_date FROM {SILVER_DB}.{table}')["rows"]]
    assert partitions == ["2026-11-01"], partitions
    print(f"OK  {table:<13} still only the partition of batch 1")

started pf-bronze-to-silver run jr_6cb4e11d201b28a317f9566f214f31e63299dfc1384b6f65f5ac539f6b4dca51


  19:45:40  RUNNING


  19:47:37  SUCCEEDED


OK  patient           34 rows = source (batch 2)


OK  encounter        870 rows = source (batch 2)


OK  condition        612 rows = source (batch 2)


OK  organization  still only the partition of batch 1


OK  practitioner  still only the partition of batch 1


## 9. A batch that was never ingested must fail clearly
There is nothing in bronze for 2026-11-09, so the job must stop with a clear message and write nothing.

In [16]:
run_4 = run_glue_job(JOB, {"--ingest_date": "2026-11-09"})
print("error:", run_4["error"])
assert run_4["state"] == "FAILED"
assert "was the batch ingested" in run_4["error"]
assert all(row[0] != "2026-11-09" for row in run_athena(f"SELECT DISTINCT ingest_date FROM {SILVER_DB}.patient")["rows"])
print("OK  the job failed with a clear message and wrote nothing")

started pf-bronze-to-silver run jr_936eed3016f1d3b56018cc04a6a88b86b28b3ecc0ffb860ae4e52d72e5e668a3


  19:48:07  RUNNING


  19:48:54  FAILED
error: RuntimeError: no patient files under s3://patientflow-datalake-zhmkbf/bronze/patient_records/ingest_date=2026-11-09/: was the batch ingested?


OK  the job failed with a clear message and wrote nothing


## 10. Job log
The job prints one `ROWCOUNT` line per table; these are in CloudWatch Logs (log group `/aws-glue/jobs/output`).

In [17]:
import time

for _ in range(6):
    events = aws_json("logs", "filter-log-events", "--log-group-name", "/aws-glue/jobs/output",
                      "--log-stream-name-prefix", run_1["run_id"], "--filter-pattern", "ROWCOUNT")["events"]
    if events:
        break
    time.sleep(10)     # logs can arrive a little after the run ends

for event in events:
    print(event["message"].strip())

ROWCOUNT table=patient ingest_date=2026-11-01 rows=35
ROWCOUNT table=encounter ingest_date=2026-11-01 rows=1411
ROWCOUNT table=condition ingest_date=2026-11-01 rows=854
ROWCOUNT table=organization ingest_date=2026-11-01 rows=521
ROWCOUNT table=practitioner ingest_date=2026-11-01 rows=521


## What this job is for

| | |
|---|---|
| Reads | `bronze/patient_records/ingest_date=<date>/` (raw FHIR JSON) |
| Writes | `silver/{patient, encounter, condition, organization, practitioner}/ingest_date=<date>/` (Parquet) |
| Used by | the gold layer (Phase 6) reads these tables; Step Functions runs this job as its second step (Phase 7) |

Silver now holds batches 1 and 2 (organization and practitioner from batch 1 only). Batches 3 and 4 are processed when the pipeline runs them.

**Next:** the gold layer: dimension and fact tables built from silver with Athena `MERGE`.